

```
# This is formatted as code
```

the notebook being mounted



In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reading the file and storing data in a data frame


In [4]:
import pandas as pd

# Indicate the location of the XLSX file on your Google Drive by specifying its path.
file_path = '/content/FallData_FD_KMP.xlsx'

#Read and store the XLSX file into a DataFrame.

df = pd.read_excel(file_path)

# Print  DataFrame
print(df)


       AAMV      IDI     MPI     MVI       PDI      ARI      FFI  SCI    Mean  \
0    1.7506  0.58000  40.216  6.3026  0.050000  0.35714   8.2746    2  10.384   
1    1.0555  0.98000  41.426  4.4719  0.040000  0.25714   8.3947    3  10.553   
2    1.5108  0.97000  30.825  5.6811  0.050000  0.44286   8.7009    3  11.372   
3    1.4725  0.89000  33.756  4.8174  0.050000  0.28571   8.4925    4  10.604   
4    2.0028  0.91000  31.238  3.1910  0.030000  0.41429   7.9257    3  10.453   
..      ...      ...     ...     ...       ...      ...      ...  ...     ...   
348  7.2999  0.51563  38.385  2.0757  0.093750  0.68889  11.7240    1  11.107   
349  6.3799  0.40625  45.453  3.7828  0.062500  0.62222   8.6792    1  10.393   
350  6.6178  0.34375  45.329  1.5182  0.046875  0.55556   6.6126    1  10.230   
351  4.7725  0.98438  55.190  3.1360  0.078125  0.55556   7.0533    3  10.739   
352  6.5607  0.54688  41.950  3.7126  0.062500  0.71111  11.9920    1  10.546   

      Median  ...     STD  

checking missing values



In [5]:
import pandas as pd

# Check missing values
missing_values = df.isnull()

# Count number of missing values in each column
missing_values_count = missing_values.sum()

# Print missing values count
print(missing_values_count)


AAMV        0
IDI         0
MPI         0
MVI         0
PDI         0
ARI         0
FFI         0
SCI         0
Mean        0
Median      0
Range       0
Variance    0
STD         0
RMS         0
SAA_x       0
SAA_y       0
SAA_z       0
SMA         0
TAV_x       0
TAV_y       0
TAV_z       0
Fall        0
dtype: int64


*data* normalization


In [6]:
for col in df.columns:
    if df[col].dtype == "float":
        df[col] = (df[col] - df[col].mean()) / df[col].std()

# Save normalized data
df.to_excel("FallData_FD_KMP_Normalized.xlsx")
print(df)

         AAMV       IDI       MPI       MVI       PDI       ARI       FFI  \
0   -0.769290 -0.453623 -0.226157  2.157734 -0.423485 -1.162820 -0.562285   
1   -0.940262  0.791712 -0.140643  0.913658 -0.819353 -1.753910 -0.534398   
2   -0.828273  0.760578 -0.889845  1.735386 -0.423485 -0.656138 -0.463298   
3   -0.837694  0.511511 -0.682703  1.148447 -0.423485 -1.585035 -0.511689   
4   -0.707258  0.573778 -0.860657  0.043206 -1.215220 -0.825012 -0.643298   
..        ...       ...       ...       ...       ...       ...       ...   
348  0.595656 -0.654029 -0.355559 -0.714710  1.308434  0.798121  0.238660   
349  0.369366 -0.994565  0.143956  0.445372  0.071349  0.404041 -0.468337   
350  0.427882 -1.189149  0.135193 -1.093566 -0.547194  0.010021 -0.948198   
351 -0.026002  0.805348  0.832096  0.005830  0.689891  0.010021 -0.845868   
352  0.413837 -0.556737 -0.103611  0.397666  0.071349  0.929461  0.300889   

     SCI      Mean    Median  ...       STD       RMS     SAA_x     SAA_y  

Selection of Features


In [7]:
from sklearn.feature_selection import f_classif

# Load data
df = pd.read_excel("FallData_FD_KMP_Normalized.xlsx")

# Get features and target variable
features = df.drop("Fall", axis=1)
target = df["Fall"]

# Select top 15 features based on their F-scores
selected_features = features.columns[f_classif(features, target)[1].argsort()[:15]]

# Print selected features
print(selected_features)

Index(['SMA', 'SAA_y', 'Unnamed: 0', 'TAV_y', 'TAV_x', 'TAV_z', 'Mean',
       'SAA_x', 'AAMV', 'ARI', 'SCI', 'Median', 'RMS', 'SAA_z', 'IDI'],
      dtype='object')


Logistic regression and kfold validation

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_excel("FallData_FD_KMP_Normalized.xlsx")

# Get features and target variable
features = df.drop("Fall", axis=1)
target = df["Fall"]

# Create k-fold cross-validation object
kf = KFold(n_splits=10)

# Loop over folds
for train_index, test_index in kf.split(features):

    # Train  model on training data
    model = LogisticRegression()
    model.fit(features.iloc[train_index], target.iloc[train_index])

    # Evaluate model on test data
    predictions = model.predict(features.iloc[test_index])
    accuracy = accuracy_score(target.iloc[test_index], predictions)

    # Print accuracy
    print("Accuracy:", accuracy)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Accuracy: 1.0
Accuracy: 1.0
Accuracy: 1.0
Accuracy: 1.0
Accuracy: 0.5428571428571428
Accuracy: 0.8
Accuracy: 1.0
Accuracy: 1.0
Accuracy: 1.0
Accuracy: 1.0


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Load the data
df = pd.read_excel("FallData_FD_KMP_Normalized.xlsx")

# Get the features and target variable
features = df.drop("Fall", axis=1)
target = df["Fall"]

# Create the k-fold cross-validation object
kf = KFold(n_splits=10)

# Loop over  folds
for train_index, test_index in kf.split(features):

    # Train model on the training data
    model = LogisticRegression()
    model.fit(features.iloc[train_index], target.iloc[train_index])

    # Evaluate  model on the test data
    predictions = model.predict(features.iloc[test_index])

    # Calculate number of features used
    num_features = len(features.columns)

    # Calculate feature selection criteria
    feature_selection_criteria = model.coef_

    # Calculate model accuracy
    accuracy = accuracy_score(target.iloc[test_index], predictions)

    # Calculate sensitivity
    sensitivity = recall_score(target.iloc[test_index], predictions)

    # Calculate specificity
    specificity = precision_score(target.iloc[test_index], predictions)

    # Print results
    print("Number of features used:", num_features)
    print("Feature selection criteria:", feature_selection_criteria)
    print("Model accuracy:", accuracy)
    print("Sensitivity:", sensitivity)
    print("Specificity:", specificity)


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parame

Number of features used: 22
Feature selection criteria: [[ 0.32675272  0.24900061 -0.29785847  0.08826836 -0.09923278  0.16023285
   0.53156223  0.18643429 -0.23933982 -0.2051144  -0.14751642 -0.00604905
   0.03046209  0.03909347 -0.10639965 -0.39841919 -0.36578751  0.03960208
  -0.40297004  0.37682199  0.01694199 -0.06246278]]
Model accuracy: 1.0
Sensitivity: 0.0
Specificity: 0.0
Number of features used: 22
Feature selection criteria: [[ 0.06463112  0.05914245 -0.08943624  0.04852972 -0.07632981 -0.16689715
   0.25932305  0.70852386 -0.28685328  0.04890979 -0.14245237 -0.20397891
  -0.04894005  0.02477591  0.02010981 -0.67862719 -0.99637265 -0.44055569
  -1.12647597  0.34570123  0.35365951 -0.01355336]]
Model accuracy: 1.0
Sensitivity: 0.0
Specificity: 0.0
Number of features used: 22
Feature selection criteria: [[ 0.04839643  0.17426531 -0.12439166  0.27371727 -0.1214376   0.0424949
   0.4102396   0.2124211  -0.24722363  0.1111244   0.22143811 -0.10942143
  -0.07354447  0.10384842  0.

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c